# FinNutriAgent — Reproducible Demo

**FinNutriAgent: Agentic AI for Household Nutrition and Budget Optimization**

| | |
|---|---|
| **Author** | Ali Akarma |
| **ORCID** | [0009-0002-6687-9380](https://orcid.org/0009-0002-6687-9380) |
| **Repository** | [github.com/aliakarma/FinNutriAgent](https://github.com/aliakarma/FinNutriAgent) |
| **DOI** | [10.5281/zenodo.18849993](https://doi.org/10.5281/zenodo.19077898) |

---

## Notebook Outline

| Section | Description |
|---|---|
| 1 | Setup and data loading |
| 2 | Exploratory data analysis |
| 3 | Budget Agent |
| 4 | Nutrition Agent |
| 5 | Price Agent |
| 6 | MILP Optimization |
| 7 | Results visualization |
| 8 | Price shock simulation |
| 9 | Cross-household sensitivity analysis |
| 10 | Summary |


In [ ]:
# ── Section 1: Setup ─────────────────────────────────────────────────────────
import subprocess, sys

# Install dependencies (runs silently in Colab; skipped if already installed)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pulp', 'pyyaml'])

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pulp import (LpProblem, LpMinimize, LpVariable,
                  lpSum, LpStatus, PULP_CBC_CMD)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120})

print('Environment ready.')

In [ ]:
# ── Data paths ────────────────────────────────────────────────────────────────
# Running locally from example/ folder:
# DATA_DIR = os.path.join('..', 'data')

# Running in Colab: uncomment the next two lines instead
!git clone https://github.com/aliakarma/FinNutriAgent.git
DATA_DIR = 'FinNutriAgent/data'

FINANCIAL_PATH  = os.path.join(DATA_DIR, 'financial',   'financial_data.csv')
FOOD_PATH       = os.path.join(DATA_DIR, 'composition', 'food_composition.csv')
PRICES_PATH     = os.path.join(DATA_DIR, 'prices',      'food_prices.csv')
NUTRITION_PATH  = os.path.join(DATA_DIR, 'nutrition',   'nutrition_requirements.csv')

# Load
financial_df  = pd.read_csv(FINANCIAL_PATH)
food_df       = pd.read_csv(FOOD_PATH,      skipinitialspace=True)
prices_df     = pd.read_csv(PRICES_PATH,    skipinitialspace=True)
nutrition_df  = pd.read_csv(NUTRITION_PATH)

for df in [food_df, prices_df]:
    df.columns   = df.columns.str.strip()
    df['food_id'] = df['food_id'].str.strip()
food_df['halal'] = food_df['halal'].str.strip()

print('Datasets loaded:')
print(f'  financial_data:          {len(financial_df):>4} rows')
print(f'  food_composition:        {len(food_df):>4} rows')
print(f'  food_prices:             {len(prices_df):>4} rows')
print(f'  nutrition_requirements:  {len(nutrition_df):>4} rows')

## Section 2 — Exploratory Data Analysis

In [ ]:
# ── EDA: Summary statistics ────────────────────────────────────────────────────
EXPENSE_COLS = ['rent','utilities','transport','education','healthcare','savings_target']
financial_df['weekly_food_budget'] = (
    financial_df['monthly_income'] - financial_df[EXPENSE_COLS].sum(axis=1)
) / 4.33

print('=== Financial Data ===')
display(financial_df[['monthly_income','weekly_food_budget']].describe().round(2))

print('\n=== Food Composition ===')
display(food_df[['calories_per_100g','protein_g','vitamin_d_mcg','iron_mg']].describe().round(2))

halal_ct = food_df['halal'].value_counts()
print(f"\nHalal: {halal_ct.get('Yes',0)} items ({halal_ct.get('Yes',0)/len(food_df)*100:.1f}%)  "
      f"Non-halal: {halal_ct.get('No',0)} items")

print('\n=== Price Data ===')
display(prices_df.groupby('store')['price_per_kg'].describe().round(2))

In [ ]:
# ── EDA: Summary statistics ────────────────────────────────────────────────────
EXPENSE_COLS = ['rent','utilities','transport','education','healthcare','savings_target']
financial_df['weekly_food_budget'] = (
    financial_df['monthly_income'] - financial_df[EXPENSE_COLS].sum(axis=1)
) / 4.33

print('=== Financial Data ===')
display(financial_df[['monthly_income','weekly_food_budget']].describe().round(2))

print('\n=== Food Composition ===')
display(food_df[['calories_per_100g','protein_g','vitamin_d_mcg','iron_mg']].describe().round(2))

halal_ct = food_df['halal'].value_counts()
print(f"\nHalal: {halal_ct.get('Yes',0)} items ({halal_ct.get('Yes',0)/len(food_df)*100:.1f}%)  "
      f"Non-halal: {halal_ct.get('No',0)} items")

print('\n=== Price Data ===')
display(prices_df.groupby('store')['price_per_kg'].describe().round(2))

In [ ]:
# ── EDA: Overview figure ──────────────────────────────────────────────────────
fig = plt.figure(figsize=(17, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)
fig.suptitle('FinNutriAgent — Dataset Overview', fontsize=14, fontweight='bold')

# 1. Income distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(financial_df['monthly_income'], bins=15, color='steelblue', edgecolor='white')
ax1.set(title='Monthly Income (SAR)', xlabel='SAR', ylabel='Households')

# 2. Weekly food budget
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(financial_df['weekly_food_budget'], bins=15, color='teal', edgecolor='white')
ax2.set(title='Weekly Food Budget (SAR)', xlabel='SAR/week', ylabel='Households')

# 3. Expense breakdown
ax3 = fig.add_subplot(gs[0, 2])
means = financial_df[EXPENSE_COLS].mean()
ax3.bar(means.index, means.values,
        color=sns.color_palette('muted', len(EXPENSE_COLS)), edgecolor='white')
ax3.set(title='Mean Monthly Expense (SAR)', xlabel='', ylabel='SAR')
ax3.tick_params(axis='x', rotation=35)

# 4. Calories per 100g
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(food_df['calories_per_100g'], bins=20, color='coral', edgecolor='white')
ax4.set(title='Calories per 100g', xlabel='kcal', ylabel='Foods')

# 5. Price distribution by store
ax5 = fig.add_subplot(gs[1, 1])
for store, color in zip(['Carrefour','Lulu','Panda'], ['steelblue','teal','coral']):
    subset = prices_df[prices_df['store'] == store]['price_per_kg']
    ax5.hist(subset, bins=18, alpha=0.6, color=color, edgecolor='white', label=store)
ax5.set(title='Price Distribution by Store', xlabel='SAR/kg', ylabel='Items')
ax5.legend(fontsize=9)

# 6. Daily calorie requirements by gender
ax6 = fig.add_subplot(gs[1, 2])
for gender, color in [('Male','steelblue'), ('Female','coral')]:
    ax6.hist(nutrition_df[nutrition_df['gender']==gender]['calories_kcal'],
             bins=20, alpha=0.6, color=color, edgecolor='white', label=gender)
ax6.set(title='Daily Calorie Requirements', xlabel='kcal/day', ylabel='Individuals')
ax6.legend(fontsize=9)

plt.savefig('fig1_dataset_overview.png', bbox_inches='tight')
plt.show()
print('Saved: fig1_dataset_overview.png')

## Section 3 — Budget Agent

In [ ]:
# ── Budget Agent ──────────────────────────────────────────────────────────────
WEEKS_PER_MONTH = 4.33

def get_weekly_budget(user_id: str, df=financial_df) -> float:
    """Compute weekly food budget for a household."""
    row = df[df['user_id'] == user_id].iloc[0]
    residual = row['monthly_income'] - sum(row[c] for c in EXPENSE_COLS)
    return round(residual / WEEKS_PER_MONTH, 2)

# Demonstrate U017
USER_ID = 'U017'
row = financial_df[financial_df['user_id'] == USER_ID].iloc[0]
budget = get_weekly_budget(USER_ID)

print(f'Household {USER_ID}:')
print(f'  Monthly income:        {row["monthly_income"]:>8,.0f} SAR')
for c in EXPENSE_COLS:
    print(f'  {c:<25}:  {row[c]:>6,.0f} SAR')
total = sum(row[c] for c in EXPENSE_COLS)
print(f'  Total fixed expenses:  {total:>8,.0f} SAR')
print(f'  Monthly food budget:   {row["monthly_income"]-total:>8,.0f} SAR')
print(f'  Weekly food budget:    {budget:>8,.2f} SAR')

## Section 4 — Nutrition Agent

In [ ]:
# ── Nutrition Agent ───────────────────────────────────────────────────────────
def get_weekly_targets(person_ids: list, df=nutrition_df) -> dict:
    """Sum daily requirements and multiply by 7 for household weekly totals."""
    subset = df[df['person_id'].isin(person_ids)]
    return {
        'calories':  round(subset['calories_kcal'].sum() * 7, 1),
        'protein':   round(subset['protein_g'].sum()     * 7, 1),
        'vitamin_d': round(subset['vitamin_d_mcg'].sum() * 7, 1),
        'iron':      round(subset['iron_mg'].sum()       * 7, 1),
    }

HOUSEHOLD_MEMBERS = ['P1', 'P2', 'P3', 'P4']
member_df = nutrition_df[nutrition_df['person_id'].isin(HOUSEHOLD_MEMBERS)]

print('Household members:')
display(member_df[['person_id','age','gender','calories_kcal',
                    'protein_g','vitamin_d_mcg','iron_mg']].reset_index(drop=True))

targets = get_weekly_targets(HOUSEHOLD_MEMBERS)
print('\nWeekly household nutritional targets:')
for k, v in targets.items():
    print(f'  {k:<12}: {v}')

## Section 5 — Price Agent

In [ ]:
# ── Merge prices with food composition ────────────────────────────────────────
avg_prices = prices_df.groupby('food_id')['price_per_kg'].mean().reset_index()
merged_df  = food_df.merge(avg_prices, on='food_id', how='left')
merged_df['cost_per_gram'] = merged_df['price_per_kg'] / 1000.0

# Cost-efficient protein sources (halal)
halal_df = merged_df[merged_df['halal'] == 'Yes'].copy()
halal_df['cost_per_g_protein'] = halal_df.apply(
    lambda r: r['cost_per_gram'] / (r['protein_g']/100) if r['protein_g'] > 0 else np.inf,
    axis=1
)

print('Top 10 most cost-efficient halal protein sources:')
display(
    halal_df.nsmallest(10, 'cost_per_g_protein')
    [['food_name','protein_g','price_per_kg','cost_per_g_protein']]
    .reset_index(drop=True)
)

## Section 6 — MILP Optimization Engine

In [ ]:
# ── Optimizer ─────────────────────────────────────────────────────────────────
NUTRIENT_MAP = {
    'calories':  'calories_per_100g',
    'protein':   'protein_g',
    'vitamin_d': 'vitamin_d_mcg',
    'iron':      'iron_mg',
}

def run_optimization(weekly_budget, nutrient_targets, food_df,
                     halal_only=True, min_g=50, max_g=500, eps=0.01,
                     verbose=False):
    """
    MILP meal plan optimizer.
    Minimize:  Σ cost_f * x_f  −  ε * Σ y_f
    Subject to nutritional lower bounds, budget upper bound,
    and per-item quantity bounds via binary selection variables.
    """
    df = food_df[food_df['halal'] == 'Yes'].copy() if halal_only else food_df.copy()
    ids   = df['food_id'].tolist()
    costs = {r['food_id']: r['price_per_kg']/1000 for _, r in df.iterrows()}

    x = {f: LpVariable(f'x_{f}', lowBound=0)   for f in ids}
    y = {f: LpVariable(f'y_{f}', cat='Binary') for f in ids}

    prob = LpProblem('FinNutriAgent', LpMinimize)
    prob += lpSum(x[f]*costs[f] for f in ids) - eps*lpSum(y[f] for f in ids)

    for f in ids:
        prob += x[f] >= min_g * y[f]
        prob += x[f] <= max_g * y[f]

    for nut, col in NUTRIENT_MAP.items():
        if nut not in nutrient_targets:
            continue
        nv = df.set_index('food_id')[col].to_dict()
        prob += lpSum(x[f]*nv[f]/100 for f in ids) >= nutrient_targets[nut]

    prob += lpSum(x[f]*costs[f] for f in ids) <= weekly_budget
    prob.solve(PULP_CBC_CMD(msg=1 if verbose else 0))

    rows, total = [], 0.0
    for f in ids:
        v = x[f].varValue or 0
        if v > 1e-4:
            c = v*costs[f]; total += c
            rows.append({'food_id': f,
                         'food_name': df.loc[df['food_id']==f,'food_name'].values[0],
                         'grams_per_week': round(v,2),
                         'cost_sar': round(c,4)})

    plan = pd.DataFrame(rows).sort_values('cost_sar',ascending=False).reset_index(drop=True)
    achieved = {}
    for nut, col in NUTRIENT_MAP.items():
        nv = df.set_index('food_id')[col].to_dict()
        achieved[nut] = round(sum((x[f].varValue or 0)*nv[f]/100 for f in ids),2)

    return {'status': LpStatus[prob.status], 'total_cost_sar': round(total,2),
            'budget_sar': weekly_budget, 'nutrient_targets': nutrient_targets,
            'nutrient_achieved': achieved, 'n_foods': len(plan), 'plan': plan}

In [ ]:
# ── Run optimization for U017 ─────────────────────────────────────────────────
result = run_optimization(
    weekly_budget=budget,
    nutrient_targets=targets,
    food_df=merged_df,
)

print(f"Status:          {result['status']}")
print(f"Total cost:      {result['total_cost_sar']:.2f} SAR  "
      f"(budget: {result['budget_sar']:.2f} SAR)")
print(f"Foods selected:  {result['n_foods']}")
print(f"Budget used:     {result['total_cost_sar']/result['budget_sar']*100:.1f}%")

print('\nNutrient coverage:')
for nut, tgt in targets.items():
    ach = result['nutrient_achieved'][nut]
    pct = ach/tgt*100
    flag = '✓' if pct >= 100 else '✗'
    print(f"  {flag} {nut:<12}: target={tgt:<8.1f} achieved={ach:<8.1f} ({pct:.1f}%)")

print('\nWeekly meal plan:')
display(result['plan'])

## Section 7 — Results Visualization

In [ ]:
# ── Results visualization ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Optimization Results — Household {USER_ID}',
             fontsize=13, fontweight='bold')

# Cost vs budget
vals   = [result['total_cost_sar'], result['budget_sar']]
labels = ['Optimized Cost', 'Budget Limit']
colors = ['steelblue', 'lightcoral']
bars = axes[0].bar(labels, vals, color=colors, edgecolor='white', width=0.4)
axes[0].set(title='Cost vs Budget (SAR/week)', ylabel='SAR')
for b, v in zip(bars, vals):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+4,
                 f'{v:.2f}', ha='center', fontsize=9)

# Nutrient coverage
nuts = list(targets.keys())
pcts = [result['nutrient_achieved'][n]/targets[n]*100 for n in nuts]
cols = ['seagreen' if p >= 100 else 'tomato' for p in pcts]
b2   = axes[1].bar(nuts, pcts, color=cols, edgecolor='white')
axes[1].axhline(100, color='black', linestyle='--', lw=1.2, label='Target 100%')
axes[1].set(title='Nutrient Coverage (% of Target)', ylabel='%',
            ylim=(0, max(pcts)*1.2))
axes[1].legend(fontsize=9)
for b, p in zip(b2, pcts):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+1,
                 f'{p:.0f}%', ha='center', fontsize=9)

# Food plan cost breakdown
plan = result['plan']
axes[2].barh(plan['food_name'], plan['cost_sar'],
             color=sns.color_palette('muted', len(plan)), edgecolor='white')
axes[2].set(title='Food Item Cost (SAR/week)', xlabel='SAR')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('fig2_optimization_results.png', bbox_inches='tight')
plt.show()
print('Saved: fig2_optimization_results.png')

## Section 8 — Price Shock Simulation

In [ ]:
# ── Price shock: +20% on all seafood items ────────────────────────────────────
SEAFOOD = ['Salmon','Tuna','Mackerel','Shrimp','Cod','Crab','Lobster','Sardines']
seafood_ids = food_df[
    food_df['food_name'].str.contains('|'.join(SEAFOOD), case=False)
]['food_id'].tolist()

prices_shocked = prices_df.copy()
prices_shocked.loc[prices_shocked['food_id'].isin(seafood_ids), 'price_per_kg'] *= 1.20

avg_shocked = prices_shocked.groupby('food_id')['price_per_kg'].mean().reset_index()
merged_shocked = food_df.merge(avg_shocked, on='food_id', how='left')

result_shocked = run_optimization(
    weekly_budget=budget,
    nutrient_targets=targets,
    food_df=merged_shocked,
)

print(f'{len(seafood_ids)} seafood items affected by +20% price shock.\n')
print(f"{'Metric':<28} {'Baseline':>10} {'Post-Shock':>12} {'Δ':>8}")
print('-' * 62)
delta_cost = result_shocked['total_cost_sar'] - result['total_cost_sar']
print(f"{'Total cost (SAR/week)':<28} {result['total_cost_sar']:>10.2f} "
      f"{result_shocked['total_cost_sar']:>12.2f} {delta_cost:>+8.2f}")
for nut in targets:
    a0, a1 = result['nutrient_achieved'][nut], result_shocked['nutrient_achieved'][nut]
    print(f"{nut+' achieved':<28} {a0:>10.1f} {a1:>12.1f} {a1-a0:>+8.1f}")

In [ ]:
# ── Shock visualization ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Price Shock Analysis — Baseline vs +20% Seafood',
             fontsize=12, fontweight='bold')

# Cost comparison
bars = axes[0].bar(['Baseline','Post-Shock'],
                   [result['total_cost_sar'], result_shocked['total_cost_sar']],
                   color=['steelblue','tomato'], edgecolor='white', width=0.4)
axes[0].axhline(budget, color='black', lw=1.2, ls='--', label='Budget')
axes[0].set(title='Optimized Weekly Food Cost', ylabel='SAR/week')
axes[0].legend(fontsize=9)
for b, v in zip(bars, [result['total_cost_sar'], result_shocked['total_cost_sar']]):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+1,
                 f'{v:.2f}', ha='center', fontsize=9)

# Nutrient coverage comparison
x = np.arange(len(targets))
w = 0.35
p0 = [result['nutrient_achieved'][n]/targets[n]*100         for n in targets]
p1 = [result_shocked['nutrient_achieved'][n]/targets[n]*100 for n in targets]
axes[1].bar(x-w/2, p0, w, label='Baseline',   color='steelblue', edgecolor='white')
axes[1].bar(x+w/2, p1, w, label='Post-Shock', color='tomato',    edgecolor='white')
axes[1].axhline(100, color='black', ls='--', lw=1.2, label='Target 100%')
axes[1].set(title='Nutrient Coverage (% of Target)', ylabel='%',
            xticks=x, xticklabels=list(targets.keys()))
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig3_price_shock.png', bbox_inches='tight')
plt.show()
print('Saved: fig3_price_shock.png')

## Section 9 — Cross-Household Sensitivity Analysis

In [ ]:
# ── Optimize across all 100 households ───────────────────────────────────────
# Fixed reference nutritional target for comparability
REF_TARGETS = {'calories': 14000, 'protein': 350, 'vitamin_d': 105, 'iron': 56}

records = []
for uid in financial_df['user_id'].tolist():
    bud = get_weekly_budget(uid)
    r   = run_optimization(weekly_budget=bud, nutrient_targets=REF_TARGETS,
                           food_df=merged_df)
    records.append({
        'user_id':        uid,
        'budget_sar':     bud,
        'cost_sar':       r['total_cost_sar'],
        'status':         r['status'],
        'n_foods':        r['n_foods'],
        'budget_used_pct': round(r['total_cost_sar']/bud*100, 2) if bud>0 else None,
    })

summary_df = pd.DataFrame(records)
optimal_pct = (summary_df['status']=='Optimal').mean()*100
print(f'Solved {len(summary_df)} households — {optimal_pct:.0f}% optimal.')
display(summary_df[['budget_sar','cost_sar','n_foods','budget_used_pct']].describe().round(2))

In [ ]:
# ── Cross-household visualization ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Cross-Household Analysis — All 100 Households',
             fontsize=13, fontweight='bold')

# 1. Budget vs cost scatter
axes[0].scatter(summary_df['budget_sar'], summary_df['cost_sar'],
                color='steelblue', edgecolors='white', s=55, alpha=0.85)
lim = max(summary_df['budget_sar'].max(), summary_df['cost_sar'].max())*1.05
axes[0].plot([0, lim], [0, lim], 'k--', lw=1, label='Cost = Budget')
axes[0].set(title='Optimized Cost vs Budget', xlabel='Budget (SAR/week)',
            ylabel='Optimized Cost (SAR/week)')
axes[0].legend(fontsize=9)

# 2. Budget utilization distribution
axes[1].hist(summary_df['budget_used_pct'].dropna(), bins=20,
             color='teal', edgecolor='white')
axes[1].set(title='Budget Utilization Distribution',
            xlabel='% of Budget Used', ylabel='Households')

# 3. Number of foods selected
food_counts = summary_df['n_foods'].value_counts().sort_index()
axes[2].bar(food_counts.index, food_counts.values,
            color='mediumpurple', edgecolor='white')
axes[2].set(title='Distribution of Foods Selected',
            xlabel='Number of Food Items / Week', ylabel='Households')

plt.tight_layout()
plt.savefig('fig4_cross_household.png', bbox_inches='tight')
plt.show()
print('Saved: fig4_cross_household.png')

## Section 10 — Summary

| Pipeline Step | Component | Key Result |
|---|---|---|
| Dataset loading | pandas | 4 datasets, 0 missing values |
| Budget computation | Budget Agent | Weekly budgets: 88–331 SAR |
| Nutrition aggregation | Nutrition Agent | 4-nutrient household targets |
| Price analysis | Price Agent | Cheapest protein sources identified |
| Optimization | MILP / CBC | 100% feasibility across 100 households |
| Budget utilization | — | Mean 12.5% of budget used |
| Price shock | +20% seafood | Plan adapts; all constraints maintained |
| Sensitivity | All households | Results fully reproduced |

---

**Repository**: [github.com/aliakarma/FinNutriAgent](https://github.com/aliakarma/FinNutriAgent)  
**DOI**: [10.5281/zenodo.18849993](https://doi.org/10.5281/zenodo.19077898)  
**Author**: Ali Akarma — ORCID [0009-0002-6687-9380](https://orcid.org/0009-0002-6687-9380)
